# 14.1 - LLMOps Foundations

**Module 14 - LLMOps** | Netsetos GenAI Engineering

This notebook works through LLMOps Foundations hands-on: What LLMOps actually is: three sets of practices; Non-determinism breaks single-point eval; Cost is a first-class metric, measured in tokens; The trace record: the foundation; Read your traces: percentiles, not averages; The LLMOps maturity model.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook is pure Python - no API keys, no network, no third-party packages. Every cell runs standalone with the standard library, so this first cell just sets the tone: the lesson uses fixed, illustrative numbers rather than live model calls.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line comment cell, not executable logic - it flags that the numbers throughout are seeded/illustrative so results are reproducible run to run. Nothing is imported here because the only non-builtin used anywhere is `json`, imported later in the trace cell.

**How the code works - step by step**
- The cell is a single comment noting the lesson works from fixed illustrative values, not random or live data.
- No imports, no state - every later cell is self-contained.

**In one line:** environment prep - the numbers are deterministic on purpose.

**What you'll see:** (no output - environment prep)

## 1 - What LLMOps actually is: three sets of practices

LLMOps is not MLOps with a new name. The whole distinction comes from one move - you *consume* a pre-trained model through an API instead of *training* your own. Model that as three sets: practices that carry over, practices that drop away, and practices that are brand new. Counting them is the fastest way to see why it is a separate discipline.

In [ ]:
# LLMOps is not MLOps renamed. When you CONSUME a model via API instead of TRAINING one, most MLOps
# machinery drops away and a new set of operational problems appears. Model it as three sets of practices.
shared     = {"deploy to prod / serving", "CI-CD pipeline", "monitoring dashboards + alerts"}
mlops_only = {"training pipeline", "feature store", "model registry (your weights)", "scheduled retraining"}
llmops_only= {"prompt versioning", "LLM-as-judge eval", "token-cost logging", "request tracing", "guardrails / safety"}
print("Shared ops practices ({}):".format(len(shared)))
for x in sorted(shared):     print("   =", x)
print("Dropped when you consume an API ({}) - no equivalent needed:".format(len(mlops_only)))
for x in sorted(mlops_only): print("   -", x)
print("Added by LLMs ({}) - NONE has an MLOps equivalent:".format(len(llmops_only)))
for x in sorted(llmops_only):print("   +", x)
print()
print("So LLMOps = MLOps  minus training  plus prompts  plus subjective eval  plus token cost.")
print("Only {} practices carry over; the {} that make an LLM app behave are all new. That is why it is its own field.".format(len(shared), len(llmops_only)))

# Output:
# Shared ops practices (3):
#    = CI-CD pipeline
#    = deploy to prod / serving
#    = monitoring dashboards + alerts
# Dropped when you consume an API (4) - no equivalent needed:
#    - feature store
#    - model registry (your weights)
#    - scheduled retraining
#    - training pipeline
# Added by LLMs (5) - NONE has an MLOps equivalent:
#    + LLM-as-judge eval
#    + guardrails / safety
#    + prompt versioning
#    + request tracing
#    + token-cost logging
#
# So LLMOps = MLOps  minus training  plus prompts  plus subjective eval  plus token cost.
# Only 3 practices carry over; the 5 that make an LLM app behave are all new. That is why it is its own field.

**Explanation**

This cell is a bookkeeping exercise over three Python sets, not a model call - it partitions ops practices into shared, MLOps-only, and LLMOps-only, then prints each with its size. The point is the count: only a handful carry over, and every genuinely new practice has no MLOps equivalent.

**How the code works - step by step**
- **`shared`** - the three practices that survive the switch: deploy/serving, CI-CD, and monitoring dashboards.
- **`mlops_only`** - the four training-centric practices that vanish when you consume an API: training pipeline, feature store, model registry, scheduled retraining.
- **`llmops_only`** - the five new practices with no MLOps twin: prompt versioning, LLM-as-judge eval, token-cost logging, request tracing, guardrails.
- **the print loop** - prints each set sorted, prefixed `=`/`-`/`+`, with its length in the header, then the one-line definition.

**In one line:** count what carries over vs. what is new -> only 3 of the practices survive, so LLMOps is its own field.

**What you'll see:** Three labelled lists: 3 shared practices, 4 dropped, 5 added - then the memorable line "LLMOps = MLOps minus training plus prompts plus subjective eval plus token cost," closing with "Only 3 practices carry over; the 5 that make an LLM app behave are all new."

## 2 - Non-determinism breaks single-point eval

The sharpest break from MLOps: a classical model with a fixed seed returns the identical output every time, so one eval suffices. Above zero temperature the same prompt returns a different answer - and often a different quality score - on every run. This cell shows why one score is a coin flip and what to report instead.

In [ ]:
# The #1 way LLMs break MLOps: NON-DETERMINISM. In classical ML a fixed seed gives one identical output, so a
# single eval suffices. With temperature > 0 the SAME prompt+input gives different outputs - so you eval statistically.
scores = [0.82, 0.79, 0.88, 0.71, 0.85, 0.80, 0.90, 0.77]  # SAME prompt+input, judged 8 times (quality 0..1), illustrative
n = len(scores)
mean = sum(scores) / n
std = (sum((s - mean) ** 2 for s in scores) / n) ** 0.5
threshold = 0.80
passes = sum(1 for s in scores if s >= threshold)
print("Same prompt, same input, run {} times (temperature > 0):".format(n))
print("  scores span {:.2f} to {:.2f}  |  mean {:.3f}  |  std {:.3f}".format(min(scores), max(scores), mean, std))
print("  pass-rate at a {:.2f} bar: {}/{} = {:.0%}".format(threshold, passes, n, passes / n))
print()
print("A SINGLE run could have reported {:.2f} (looks failing) or {:.2f} (looks great) - a coin flip.".format(min(scores), max(scores)))
print("So an LLM eval is a DISTRIBUTION, not a point: report mean +/- std and pass-rate over N runs, never one call.")
print("Classical ML with a fixed seed reproduces one output, so one eval is enough. That assumption is gone here.")

# Output:
# Same prompt, same input, run 8 times (temperature > 0):
#   scores span 0.71 to 0.90  |  mean 0.815  |  std 0.058
#   pass-rate at a 0.80 bar: 5/8 = 62%
#
# A SINGLE run could have reported 0.71 (looks failing) or 0.90 (looks great) - a coin flip.
# So an LLM eval is a DISTRIBUTION, not a point: report mean +/- std and pass-rate over N runs, never one call.
# Classical ML with a fixed seed reproduces one output, so one eval is enough. That assumption is gone here.

**Explanation**

A small statistics harness over a hardcoded list of eight judge scores for the *same* prompt run eight times - no LLM is actually called, the scores are illustrative. It computes the mean, standard deviation, and pass-rate to make the point that an LLM eval is a distribution, not a point.

**How the code works - step by step**
- **`scores`** - eight quality scores (0..1) for the same prompt+input, judged eight times.
- **`mean` / `std`** - population mean and standard deviation computed by hand from the list.
- **`threshold` / `passes`** - counts how many runs clear a 0.80 quality bar to get a pass-rate.
- **the prints** - report the span (min..max), mean, std, the pass-rate fraction, and the punchline that a single run could have shown the min (looks failing) or the max (looks great).

**In one line:** run the same input N times -> summarise with mean, spread, and pass-rate, never one score.

**What you'll see:** Scores spanning 0.71 to 0.90 with mean 0.815 and std 0.058, a pass-rate of 5/8 = 62% at the 0.80 bar, and the warning that a single run could have reported 0.71 or 0.90 - a coin flip.

## 3 - Cost is a first-class metric, measured in tokens

In MLOps cost meant compute hours and was predictable. In LLMOps cost is measured in tokens - input and output each priced separately - and the *same* feature on the *same* model can cost a hundred times more on one call than another. This cell prices two requests to show the spread and where it comes from.

In [ ]:
# Cost is a FIRST-CLASS metric in LLMOps (it never was in MLOps). The unit is the TOKEN, and the SAME feature on
# the SAME model can cost 100x more on one request than another - so you must LOG tokens per call or you fly blind.
PRICE_IN, PRICE_OUT = 3.00, 15.00                      # $ per 1M tokens, illustrative sonnet-class rates
def cost(in_tok, out_tok):
    return (in_tok * PRICE_IN + out_tok * PRICE_OUT) / 1_000_000
requests = [("short classify", 50, 8), ("RAG answer (retrieved context stuffed in)", 8000, 400)]
costs = {}
for name, i, o in requests:
    c = cost(i, o); costs[name] = c
    print("  {:<42} {:>5} in + {:>4} out tokens -> ${:.6f}".format(name, i, o, c))
ratio = costs["RAG answer (retrieved context stuffed in)"] / costs["short classify"]
print()
print("Same model, same feature: the RAG request costs about {:.0f}x the classify request - all from tokens.".format(ratio))
print("The retrieved context (input) dominates. If a trace does not record input_tokens and output_tokens per call,")
print("you cannot see spend, cannot attribute it, cannot cut it. Logging the tokens comes first; optimising comes later (M13, 14.5).")

# Output:
#   short classify                                50 in +    8 out tokens -> $0.000270
#   RAG answer (retrieved context stuffed in)   8000 in +  400 out tokens -> $0.030000
#
# Same model, same feature: the RAG request costs about 111x the classify request - all from tokens.
# The retrieved context (input) dominates. If a trace does not record input_tokens and output_tokens per call,
# you cannot see spend, cannot attribute it, cannot cut it. Logging the tokens comes first; optimising comes later (M13, 14.5).

**Explanation**

A tiny pricing function applied to two illustrative requests - a measurement, not a model call. It multiplies token counts by per-million rates to show that the retrieval-augmented request costs about a hundred times the short classify request, driven almost entirely by input context.

**How the code works - step by step**
- **`PRICE_IN, PRICE_OUT`** - illustrative sonnet-class rates in dollars per million tokens ($3 in, $15 out).
- **`cost(in_tok, out_tok)`** - the pricing formula: (input x in-price + output x out-price) / 1M.
- **`requests`** - two cases: a short classify (50 in / 8 out) and a RAG answer that stuffs 8000 tokens of context in (8000 in / 400 out).
- **the loop** - prices each request, stores it in `costs`, and prints the per-request breakdown.
- **`ratio`** - divides the RAG cost by the classify cost to get the multiple.

**In one line:** cost = tokens x price; the big-context request dominates, so you must log tokens per call.

**What you'll see:** The classify call at $0.000270 and the RAG call at $0.030000, with the note that the RAG request costs about 111x the classify request - all from tokens, mostly the input context.

## 4 - The trace record: the foundation

Everything so far - token cost, statistical eval, debugging a complaint - needs the same thing underneath: a record of what happened on each call. That record is the foundational LLMOps artifact, and it is smaller than people expect: one instrumented call emits one structured JSON line. Its field names follow the OpenTelemetry GenAI conventions so any backend can read it.

In [ ]:
# The foundational LLMOps artifact: one INSTRUMENTED call that emits a structured trace record. Field names map to
# the OpenTelemetry GenAI semantic conventions (the 2026 vendor-neutral standard) so any backend can read them.
import json
PRICE = {"claude-sonnet-4-6": {"in": 3.00, "out": 15.00}}   # illustrative $/1M
def make_trace(call_id, ts, user_id, feature, model, prompt_version, in_tok, out_tok, latency_ms, error, prompt, output):
    p = PRICE.get(model, {"in": 0, "out": 0})
    return {
        "call_id": call_id, "ts": ts, "user_id": user_id, "feature": feature,
        "gen_ai.request.model": model, "prompt_version": prompt_version,
        "gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok,
        "cost_usd": round((in_tok * p["in"] + out_tok * p["out"]) / 1_000_000, 6),
        "latency_ms": latency_ms, "error": error,
        "prompt_preview": prompt[:50], "output_preview": (output or "")[:50],
    }
# token counts + latency come off the response object; here they are given, so this runs with no network or key.
trace = make_trace("c-0001", "2026-07-10T14:03:07Z", "u_421", "ticket-classifier",
                   "claude-sonnet-4-6", "clf-v3", 62, 5, 240, None,
                   "Ticket: pizza arrived cold, refund please", "refund")
print(json.dumps(trace, indent=2))
print("# One JSON line per call, appended to a .jsonl file. THIS record is the whole foundation of LLM observability.")
print("# LangSmith, Langfuse, and Phoenix are this record plus a UI and aggregation - you meet them in Lesson 14.3.")

# Output:
# {
#   "call_id": "c-0001",
#   "ts": "2026-07-10T14:03:07Z",
#   "user_id": "u_421",
#   "feature": "ticket-classifier",
#   "gen_ai.request.model": "claude-sonnet-4-6",
#   "prompt_version": "clf-v3",
#   "gen_ai.usage.input_tokens": 62,
#   "gen_ai.usage.output_tokens": 5,
#   "cost_usd": 0.000261,
#   "latency_ms": 240,
#   "error": null,
#   "prompt_preview": "Ticket: pizza arrived cold, refund please",
#   "output_preview": "refund"
# }
# # One JSON line per call, appended to a .jsonl file. THIS record is the whole foundation of LLM observability.
# # LangSmith, Langfuse, and Phoenix are this record plus a UI and aggregation - you meet them in Lesson 14.3.

**Explanation**

A factory function that assembles one trace as a plain dict and prints it as JSON - the single most important artifact in the module. Token counts and latency would come off a real response object; here they are passed in, so it runs with no key or network. The `gen_ai.*` keys are the vendor-neutral OpenTelemetry names.

**How the code works - step by step**
- **`import json`** - the only non-builtin dependency in the notebook, used for pretty-printing.
- **`PRICE`** - a per-model rate table (illustrative $/1M) used to compute cost inside the trace.
- **`make_trace(...)`** - builds the record: identity (`call_id`, `ts`, `user_id`, `feature`), the `gen_ai.request.model` and `prompt_version`, the `gen_ai.usage.*` token counts, a computed `cost_usd`, `latency_ms`, `error`, and 50-char previews of prompt and output.
- **the call + `json.dumps`** - builds one sample trace for a ticket-classifier call and prints it indented.

**In one line:** one call -> one structured JSON line carrying everything you need to debug, price, and group it.

**What you'll see:** A single indented JSON object with call_id `c-0001`, model `claude-sonnet-4-6`, 62 input / 5 output tokens, a computed `cost_usd` of 0.000261, 240 ms latency, and null error - followed by notes that this record is the whole foundation and that LangSmith/Langfuse/Phoenix are just this plus a UI.

## 5 - Read your traces: percentiles, not averages

A pile of trace records is only useful if you read it, and the instinct to look at the average is exactly what misleads you. The mean latency hides both the honest typical request (p50) and the painful tail (p95) an unlucky user feels; and a few expensive calls often drive most of the spend. This cell aggregates a day of traces the right way.

In [ ]:
# Once every call emits a trace, you become the analyst of your own logs. The insight is in the PERCENTILES, not the
# average: the mean hides both the honest typical (p50) and the painful tail (p95) - and a few calls drive most spend.
classify = [{"lat": l, "cost": 0.00026, "err": None, "feat": "classify"}
            for l in [180,185,190,195,200,205,210,215,220,225,230,235,240]]
rag = [{"lat": l, "cost": 0.030, "err": None, "feat": "rag_answer"}
       for l in [260,280,300,900,950,1100,1300]]
traces = classify + rag
traces[10]["err"] = "rate_limit"                    # one failed call among the 20
n = len(traces)
lats = sorted(t["lat"] for t in traces)
def pct(vals, p):                                   # nearest-rank percentile
    k = max(1, -(-p * len(vals) // 100))            # ceil(p/100 * n)
    return vals[k - 1]
mean_lat = sum(lats) / n
total_cost = sum(t["cost"] for t in traces)
errors = sum(1 for t in traces if t["err"])
rag_cost = sum(t["cost"] for t in traces if t["feat"] == "rag_answer")
print("{} traces:".format(n))
print("  latency  mean {:.0f} ms   p50 {} ms   p95 {} ms   (the mean hides both)".format(mean_lat, pct(lats,50), pct(lats,95)))
print("  errors   {}/{} = {:.0%}".format(errors, n, errors / n))
print("  cost     ${:.4f} total   rag_answer = ${:.4f} ({:.0%} of spend from {:.0%} of calls)".format(
      total_cost, rag_cost, rag_cost / total_cost, sum(1 for t in traces if t["feat"]=="rag_answer") / n))
print()
print("p95 ({} ms) is about {:.0f}x the p50 ({} ms) - that tail is what a user actually feels, and the mean buries it.".format(
      pct(lats,95), pct(lats,95) / pct(lats,50), pct(lats,50)))
print("A third of the calls (RAG) drive almost all the cost. You could not see any of this without the trace records.")

# Output:
# 20 traces:
#   latency  mean 391 ms   p50 225 ms   p95 1100 ms   (the mean hides both)
#   errors   1/20 = 5%
#   cost     $0.2134 total   rag_answer = $0.2100 (98% of spend from 35% of calls)
#
# p95 (1100 ms) is about 5x the p50 (225 ms) - that tail is what a user actually feels, and the mean buries it.
# A third of the calls (RAG) drive almost all the cost. You could not see any of this without the trace records.

**Explanation**

A trace analyzer over twenty synthetic records - it computes percentiles, error rate, and per-feature cost attribution rather than single averages. It exists to prove that the mean latency buries the tail and that a small slice of calls dominates cost, both invisible without the per-call records from the previous cell.

**How the code works - step by step**
- **`classify` / `rag`** - two batches of trace dicts (latency, cost, error, feature); RAG calls are far slower and far pricier.
- **`traces[10]["err"]`** - injects one `rate_limit` failure among the twenty calls.
- **`pct(vals, p)`** - a nearest-rank percentile helper using a ceiling of p/100 x n.
- **`mean_lat` / `total_cost` / `errors` / `rag_cost`** - the aggregate stats, including cost attributed to the `rag_answer` feature.
- **the prints** - report mean vs. p50 vs. p95 latency, the error rate, total and RAG cost with its share, and that p95 is about 5x p50.

**In one line:** aggregate the trace file into percentiles and per-feature cost - the mean alone lies.

**What you'll see:** For 20 traces: mean 391 ms but p50 225 ms and p95 1100 ms, a 5% error rate, and $0.2134 total cost where RAG is $0.2100 - 98% of spend from 35% of calls. The p95 is about 5x the p50, the tail the mean buries.

## 6 - The LLMOps maturity model

Zoom out from one call to a whole team. Maturity is a ladder of five levels - Vibes, Logs, Eval, CI, Continuous - each gated by two capabilities. Your true level is the highest rung for which *every* capability up to it is met; you cannot skip a rung. This cell scores a sample team and names the single next rung to buy.

In [ ]:
# The LLMOps maturity model: 5 levels, each gated by two capabilities. A team's level is the highest rung for which
# EVERY capability up to it is met - you cannot skip a rung. Going 0 -> 1 is the biggest single jump.
caps = {
    "prompts_in_git": True,  "structured_traces": True,     # Level 1  Logs
    "golden_set": True,      "eval_on_change": False,       # Level 2  Eval
    "ci_eval_gate": False,   "cost_budget": False,          # Level 3  CI
    "live_eval": False,      "auto_rollback": False,        # Level 4  Continuous
}
LEVELS = [(1,"Logs",["prompts_in_git","structured_traces"]),
          (2,"Eval",["golden_set","eval_on_change"]),
          (3,"CI",["ci_eval_gate","cost_budget"]),
          (4,"Continuous",["live_eval","auto_rollback"])]
level, next_gap = 0, None
for num, name, req in LEVELS:
    met = [c for c in req if caps[c]]
    status = "MET" if len(met) == len(req) else "partial ({}/{})".format(len(met), len(req))
    print("  Level {} {:<11} needs {:<40} -> {}".format(num, name, ", ".join(req), status))
    if level == num - 1 and len(met) == len(req):
        level = num
    elif next_gap is None and len(met) < len(req):
        next_gap = (num, name, [c for c in req if not caps[c]])
print()
print("This team is at Level {}. The next dollar buys Level {} ({}): add {}.".format(
      level, next_gap[0], next_gap[1], " + ".join(next_gap[2])))
print("Aim ONE rung up, not four. 0 -> 1 (print-and-pray to structured traces + prompts in git) is the biggest win of all.")

# Output:
#   Level 1 Logs        needs prompts_in_git, structured_traces        -> MET
#   Level 2 Eval        needs golden_set, eval_on_change               -> partial (1/2)
#   Level 3 CI          needs ci_eval_gate, cost_budget                -> partial (0/2)
#   Level 4 Continuous  needs live_eval, auto_rollback                 -> partial (0/2)
#
# This team is at Level 1. The next dollar buys Level 2 (Eval): add eval_on_change.
# Aim ONE rung up, not four. 0 -> 1 (print-and-pray to structured traces + prompts in git) is the biggest win of all.

**Explanation**

A scorer that walks a fixed ladder against a team's capability flags - it climbs only while each rung is fully met and records the first gap. The logic encodes the no-skipping rule: partial completion of a higher rung does not raise your level, and the output points at exactly one rung up.

**How the code works - step by step**
- **`caps`** - eight boolean capabilities grouped by the level they gate (Logs, Eval, CI, Continuous).
- **`LEVELS`** - the ordered ladder: each entry is (number, name, list of required capability keys).
- **the loop** - for each rung, checks which required caps are met, prints MET or partial(m/n); promotes `level` only when the current rung is the next one up AND fully met; records the first partial rung as `next_gap`.
- **the closing prints** - state the team's level, the cheapest next rung and the exact missing capability, and the rule to aim one rung up.

**In one line:** climb the ladder rung by rung - your level is the highest fully-met rung, and 0->1 is the biggest jump.

**What you'll see:** Level 1 Logs marked MET, Level 2 Eval partial (1/2), Levels 3-4 partial (0/2), concluding the team is at Level 1 and the next dollar buys Level 2 by adding `eval_on_change`.

## 7 - Match the investment to the stakes

The final question decides how much of all this to build: how much LLMOps does *this* project need? The answer is a function of scale (users and cost) and stakes (what happens when it is wrong) - and stakes can override volume entirely. This cell runs the decision over four scenarios, including a low-volume, high-stakes clinical bot.

In [ ]:
# When do you need LLMOps, and how much? Match the investment to the STAKES, not to your enthusiasm. Over-building a
# demo wastes weeks; under-building a product is negligence; high stakes force the full stack regardless of volume.
def recommend(users, monthly_usd, high_stakes):
    if high_stakes:                                   # medical / legal / financial: stakes override volume
        return "FULL stack + guardrails + human-in-the-loop + audit trail"
    if users < 50 and monthly_usd < 50:
        return "Level 0 - structured logs to stdout, move on"
    if users < 5000 and monthly_usd < 5000:
        return "Level 1-2 - tracing + prompt versioning + a small golden set"
    return "Level 3-4 - full stack: CI eval gate + continuous eval + alerts"
scenarios = [("hackathon demo", 8, 20, False),
             ("company-wide internal tool", 2000, 3000, False),
             ("customer-facing feature", 40000, 30000, False),
             ("clinical triage assistant", 500, 4000, True)]
for name, users, spend, stakes in scenarios:
    print("  {:<28} {:>6} users  ${:>6}/mo  stakes={:<5} -> {}".format(name, users, spend, str(stakes), recommend(users, spend, stakes)))
print()
print("The two failure modes: a 50-user demo carrying a full CI-eval stack (wasted), and a 40,000-user feature shipped")
print("on print statements (a fire waiting to happen). And note the clinical bot: LOW volume, but stakes force the full stack.")

# Output:
#   hackathon demo                    8 users  $    20/mo  stakes=False -> Level 0 - structured logs to stdout, move on
#   company-wide internal tool     2000 users  $  3000/mo  stakes=False -> Level 1-2 - tracing + prompt versioning + a small golden set
#   customer-facing feature       40000 users  $ 30000/mo  stakes=False -> Level 3-4 - full stack: CI eval gate + continuous eval + alerts
#   clinical triage assistant       500 users  $  4000/mo  stakes=True  -> FULL stack + guardrails + human-in-the-loop + audit trail
#
# The two failure modes: a 50-user demo carrying a full CI-eval stack (wasted), and a 40,000-user feature shipped
# on print statements (a fire waiting to happen). And note the clinical bot: LOW volume, but stakes force the full stack.

**Explanation**

A decision function encoding the right-sizing rule, run over four scenarios - a rules table, not a model. The key branch is that `high_stakes` short-circuits to the full stack before any volume check, so a small clinical tool still gets guardrails and human-in-the-loop while a big consumer feature scales up on volume.

**How the code works - step by step**
- **`recommend(users, monthly_usd, high_stakes)`** - the rule: high-stakes returns the full stack immediately; otherwise Level 0 for tiny/cheap, Level 1-2 for mid-scale, Level 3-4 (full stack) above that.
- **`scenarios`** - four cases: hackathon demo, company internal tool, customer-facing feature, and a low-volume high-stakes clinical assistant.
- **the loop** - prints each scenario's users, monthly spend, stakes flag, and recommended level.
- **the closing prints** - name the two symmetric failure modes (over-building a demo, under-building a product) and flag that the clinical bot's low volume still forces the full stack.

**In one line:** size the ops to scale AND stakes - and stakes override volume every time.

**What you'll see:** Four recommendations: the demo -> Level 0, the internal tool -> Level 1-2, the customer feature -> Level 3-4 full stack, and the clinical bot (only 500 users) -> full stack plus guardrails, human-in-the-loop, and audit trail because stakes override volume.

Across seven cells you built the four foundational LLMOps tools by hand: the three-set comparison that defines the field, two demonstrations of the first-class breaks (non-determinism forces a distribution, token cost swings a hundredfold), the structured trace record that carries it all, a trace analyzer that computes percentiles and cost attribution, a maturity-model scorer, and an investment-decision function. The through-line is the trace record - it is the raw material the analyzer reads, the maturity ladder assumes, and every later lesson in Module 14 builds on. Next up is Lesson 14.2, which versions the prompt that made each traced call behave.